In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number, desc, lit, concat
from pyspark.sql.window import Window

POSTGRES_JAR = "/home/jovyan/jars/postgresql-42.7.4.jar"
CLICKHOUSE_JAR = "/home/jovyan/jars/clickhouse-jdbc-0.9.8-all-dependencies.jar"

spark = SparkSession.builder \
    .appName("FillClickHouseReports") \
    .config("spark.jars", f"{POSTGRES_JAR},{CLICKHOUSE_JAR}") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

PG_URL = "jdbc:postgresql://postgres:5432/postgres"
PG_PROPS = {
    "user": "postgres",
    "password": "POSTGRES_PASSWORD",
    "driver": "org.postgresql.Driver"
}

CH_URL = "jdbc:clickhouse://clickhouse:8123/default"
CH_PROPS = {
    "user": "default",
    "password": "clickhouse123",
    "driver": "com.clickhouse.jdbc.ClickHouseDriver"
}

In [2]:
dim_product = spark.read.jdbc(PG_URL, "star.dim_product", properties=PG_PROPS)
dim_customer = spark.read.jdbc(PG_URL, "star.dim_customer", properties=PG_PROPS)
dim_date = spark.read.jdbc(PG_URL, "star.dim_date", properties=PG_PROPS)
dim_store = spark.read.jdbc(PG_URL, "star.dim_store", properties=PG_PROPS)
dim_supplier = spark.read.jdbc(PG_URL, "star.dim_supplier", properties=PG_PROPS)
fact_sales = spark.read.jdbc(PG_URL, "star.fact_sales", properties=PG_PROPS)

dim_product.cache()
dim_customer.cache()
dim_date.cache()
dim_store.cache()
dim_supplier.cache()
fact_sales.cache()

print("Данные загружены:")
print(f"  fact_sales: {fact_sales.count()}")
print(f"  dim_product: {dim_product.count()}")
print(f"  dim_customer: {dim_customer.count()}")
print(f"  dim_date: {dim_date.count()}")
print(f"  dim_store: {dim_store.count()}")
print(f"  dim_supplier: {dim_supplier.count()}")


Данные загружены:
  fact_sales: 10000
  dim_product: 1000
  dim_customer: 1000
  dim_date: 364
  dim_store: 383
  dim_supplier: 383


In [3]:
report_top_products = (fact_sales.groupBy("product_id")
                       .agg(spark_sum("quantity").alias("total_quantity"))
                       .join(dim_product.select("product_id", "product_name"), on="product_id")
                       .orderBy(desc("total_quantity"))
                       .limit(10)
                       .withColumn("rank", row_number().over(Window.orderBy(desc("total_quantity"))))
                       .select("rank", "product_id", "product_name", "total_quantity"))

report_top_customers = (fact_sales.groupBy("customer_id")
                        .agg(spark_sum("total_price").alias("total_spent"))
                        .join(dim_customer.select("customer_id", "first_name", "last_name"), on="customer_id")
                        .withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name")))
                        .orderBy(desc("total_spent"))
                        .limit(10)
                        .withColumn("rank", row_number().over(Window.orderBy(desc("total_spent"))))
                        .select("rank", "customer_id", "full_name", "total_spent"))

fact_with_date = fact_sales.join(dim_date, on="date_id")
report_monthly_trends = (fact_with_date
                         .groupBy("year", "month", "month_name")
                         .agg(spark_sum("total_price").alias("total_revenue"),
                              spark_sum("quantity").alias("total_quantity"),
                              count("sale_id").alias("num_orders"))
                         .orderBy("year", "month")
                         .select("year", "month", "month_name", "total_revenue", "total_quantity", "num_orders"))

fact_with_store = fact_sales.join(dim_store, on="store_id")
report_top_stores = (fact_with_store
                     .groupBy("store_id", "store_name")
                     .agg(spark_sum("total_price").alias("total_revenue"))
                     .orderBy(desc("total_revenue"))
                     .limit(5)
                     .withColumn("rank", row_number().over(Window.orderBy(desc("total_revenue"))))
                     .select("rank", "store_id", "store_name", "total_revenue"))

fact_with_supplier = (fact_sales
                      .join(dim_product, on="product_id")
                      .join(dim_supplier, on="supplier_id"))
report_top_suppliers = (fact_with_supplier
                        .groupBy("supplier_id", "supplier_name")
                        .agg(spark_sum("total_price").alias("total_revenue"))
                        .orderBy(desc("total_revenue"))
                        .limit(5)
                        .withColumn("rank", row_number().over(Window.orderBy(desc("total_revenue"))))
                        .select("rank", "supplier_id", "supplier_name", "total_revenue"))

max_rating = (dim_product.orderBy(desc("product_rating"))
              .limit(1)
              .withColumn("rating_type", lit("Highest"))
              .select("rating_type", "product_id", "product_name", col("product_rating").alias("product_rating")))

min_rating = (dim_product.orderBy("product_rating")
              .limit(1)
              .withColumn("rating_type", lit("Lowest"))
              .select("rating_type", "product_id", "product_name", col("product_rating").alias("product_rating")))

report_extreme_ratings = max_rating.union(min_rating)



In [4]:

from urllib import request, parse

def clear_clickhouse_table(table_name):
    url = "http://clickhouse:8123/"
    params = {"password": "clickhouse123"}
    query = f"TRUNCATE TABLE {table_name}".encode('utf-8')
    req = request.Request(url + "?" + parse.urlencode(params), data=query, method='POST')
    try:
        with request.urlopen(req) as resp:
            if resp.status == 200:
                print(f"Таблица {table_name} очищена")
            else:
                print(f"Ошибка: {resp.read().decode()}")
    except Exception as e:
        print(f"Не удалось очистить {table_name}: {e}")

def write_to_clickhouse(df, table_name):
    clear_clickhouse_table(table_name)
    df.write.jdbc(url=CH_URL, table=table_name, mode="append", properties=CH_PROPS)
    print(f"Записано {df.count()} строк в {table_name}")


print("Начинаем запись отчётов в ClickHouse...")
write_to_clickhouse(report_top_products, "top_products")
write_to_clickhouse(report_top_customers, "top_customers")
write_to_clickhouse(report_monthly_trends, "monthly_trends")
write_to_clickhouse(report_top_stores, "top_stores")
write_to_clickhouse(report_top_suppliers, "top_suppliers")
write_to_clickhouse(report_extreme_ratings, "extreme_ratings")
print("Все отчёты успешно записаны!")

test_df = spark.read.jdbc(CH_URL, "top_products", properties=CH_PROPS)
test_df.show(10, truncate=False)

spark.stop()

Начинаем запись отчётов в ClickHouse...
Таблица top_products очищена
Записано 10 строк в top_products
Таблица top_customers очищена
Записано 10 строк в top_customers
Таблица monthly_trends очищена
Записано 12 строк в monthly_trends
Таблица top_stores очищена
Записано 5 строк в top_stores
Таблица top_suppliers очищена
Записано 5 строк в top_suppliers
Таблица extreme_ratings очищена
Записано 2 строк в extreme_ratings
Все отчёты успешно записаны!
+----+----------+------------+--------------+
|rank|product_id|product_name|total_quantity|
+----+----------+------------+--------------+
|1   |562       |Bird Cage   |84            |
|2   |963       |Dog Food    |84            |
|3   |673       |Bird Cage   |80            |
|4   |692       |Cat Toy     |80            |
|5   |392       |Bird Cage   |78            |
|6   |622       |Cat Toy     |77            |
|7   |624       |Cat Toy     |77            |
|8   |699       |Bird Cage   |77            |
|9   |790       |Bird Cage   |77            |
